[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helvecioneto/datascience/blob/main/praticas/pratica03-2-arvore-decisao.ipynb)

# Aula Prática 03 · Exemplo 2 de 3: Árvore de Decisão

**Disciplina:** Introdução à Ciência de Dados (NSA A110) · **Professor:** Helvecio Bezerra Leal Neto · UFOPA
**Data:** 29/10/2026, 19h às 22h · **Modalidade:** prática em equipes (2 a 3 pessoas)

Hoje são **três notebooks curtos**, um para cada técnica: **1.** clusterização com DBSCAN · **2.** árvore de decisão · **3.** redes neurais. Este é o **2**.

### Como usar este notebook
Leia cada bloco de texto **antes** de rodar a célula de código logo abaixo dele. Os ícones dizem o que fazer:

| Ícone | O que você faz |
|---|---|
| ▶️ **Rode** | A célula está pronta. Execute com `Shift + Enter` e leia a saída. |
| 🧭 **Complete** | Troque cada `___` pelo que o texto pede e rode. Se travar, abra o **💡 Ver resposta** logo abaixo, mas tente antes. |
| ✍️ **Responda** | Escreva a resposta da equipe na célula de texto indicada (dois cliques para editar). |

Rode as células **na ordem**, de cima para baixo. Se algo der errado: *Ambiente de execução → Reiniciar e executar tudo*.

### Objetivos
Ao final, a equipe será capaz de:
- **Treinar** uma árvore de decisão e **ler** as regras dela em português.
- **Usar** a árvore para prever casos novos.
- **Avaliar** o modelo no teste, comparando com uma linha de base, e **escolher** a profundidade da árvore sem cair no sobreajuste.

### O problema: quem sobreviveu ao Titanic?

Em abril de 1912 o navio **Titanic** bateu num iceberg e afundou. Não havia botes salva-vidas para todos, e a regra da época era *"mulheres e crianças primeiro"*.

Vamos usar a lista **real** de passageiros para responder:

> Sabendo a **classe**, o **sexo**, a **idade** e o **preço da passagem** de um passageiro, conseguimos prever se ele **sobreviveu**?

É um problema de **aprendizado supervisionado** (Aula 07): temos o **rótulo** (sobreviveu ou não) e queremos aprender a prevê-lo.

**A árvore de decisão** aprende uma sequência de **perguntas de sim ou não**, como no exemplo do spam da Aula 07 (*"tem palavras suspeitas?"*). Aqui as perguntas serão do tipo *"é homem?"* ou *"tem menos de 10 anos?"*. O algoritmo testa as perguntas possíveis e fica com a que **melhor separa** quem sobreviveu de quem morreu; depois repete dentro de cada resposta.

## Etapa 0 · Preparar o ambiente
▶️ **Rode.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

SEMENTE = 42   # mesma semente = mesmos resultados para todas as equipes
print("Tudo pronto!")

## Etapa 1 · Carregar os dados
▶️ **Rode.** A célula baixa a lista de passageiros, guarda só 5 colunas, traduz os nomes e **tira os passageiros sem idade registrada** (a árvore precisa de todos os valores preenchidos).

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
titanic = pd.read_csv(URL)

colunas = {"survived": "sobreviveu", "pclass": "classe", "sex": "sexo", "age": "idade", "fare": "tarifa"}
df = titanic[list(colunas)].rename(columns=colunas)

print("Passageiros na lista:", len(df))
print("Passageiros sem idade registrada:", df["idade"].isna().sum())
df = df.dropna().reset_index(drop=True)
print("Passageiros que vamos usar:", len(df))
df.head()

O que é cada coluna:

| Coluna | Significado |
|---|---|
| `sobreviveu` | **1** = sobreviveu · **0** = morreu. É o **rótulo** (o **y**). |
| `classe` | classe da passagem: **1** (primeira, a mais cara), **2** ou **3** |
| `sexo` | `female` (mulher) ou `male` (homem) |
| `idade` | idade em anos |
| `tarifa` | preço pago pela passagem (em libras) |

As quatro últimas são as **entradas** (o **X**).

## Etapa 2 · Conhecer os dados
### Quantos sobreviveram?
🧭 **Complete** com o método do pandas que **conta quantas vezes cada valor aparece** numa coluna. Com `normalize=True` ele mostra proporções.

In [ ]:
df["sobreviveu"].___(normalize=True).round(3)

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
df["sobreviveu"].value_counts(normalize=True).round(3)
```
</details>

✍️ **Responda:** imagine um "modelo preguiçoso" que **não olha nada** e responde sempre *"morreu"*. Que proporção de passageiros ele acertaria? Esse número é a **linha de base**: a árvore precisa ser **melhor** que ele.

**✍️ Resposta da equipe:** a linha de base é de ____ %.

### Quem tinha mais chance?
Truque útil: numa coluna de **0 e 1**, a **média** é a **proporção de 1s**. Então a média de `sobreviveu` num grupo é a **taxa de sobrevivência** daquele grupo.

🧭 **Complete** com o nome da coluna pela qual queremos separar os passageiros: o **sexo**.

In [ ]:
df.groupby("___")["sobreviveu"].mean().round(3)

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
df.groupby("sexo")["sobreviveu"].mean().round(3)
```
</details>

▶️ **Rode** para ver a taxa de sobrevivência separada por **classe e sexo** ao mesmo tempo.

In [ ]:
taxa = df.groupby(["classe", "sexo"])["sobreviveu"].mean().unstack()
taxa.plot(kind="bar", rot=0, figsize=(7, 4), title="Taxa de sobrevivência por classe e sexo")
plt.ylabel("proporção que sobreviveu")
plt.show()
taxa.round(2)

✍️ **Responda:**
1. Quem tinha mais chance de sobreviver: homens ou mulheres? Quanto, em cada caso?
2. A classe da passagem também importava? Deem um exemplo com números do gráfico.

**✍️ Respostas da equipe**

1.
2.

## Etapa 3 · Preparar os dados para o modelo
A árvore do scikit-learn só trabalha com **números**, e a coluna `sexo` tem **texto**. Vamos trocar `female` por **1** e `male` por **0**.

🧭 **Complete** o dicionário do `map`: mulher vale **1**, homem vale **0**.

In [ ]:
df["sexo"] = df["sexo"].map({"female": ___, "male": ___})
df.head()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
df["sexo"] = df["sexo"].map({"female": 1, "male": 0})
```
</details>

Agora separamos **X** (entradas) e **y** (rótulo) e dividimos em **treino (75%)** e **teste (25%)**, como na Aula 07: o modelo aprende com o treino e é avaliado no teste, com passageiros que **nunca viu**.

▶️ **Rode.**

In [ ]:
X = df[["classe", "sexo", "idade", "tarifa"]]   # entradas
y = df["sobreviveu"]                            # rótulo

X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEMENTE
)
print("Treino:", len(X_treino), "passageiros")
print("Teste: ", len(X_teste), "passageiros")

## Etapa 4 · Treinar uma árvore pequena
Vamos começar com uma árvore de **profundidade 2**: cada passageiro responde a **no máximo 2 perguntas** antes da resposta final. Assim conseguimos ler a árvore inteira.

Treinar, no scikit-learn, é sempre o mesmo gesto: `modelo.fit(X, y)`.

🧭 **Complete** com as entradas e os rótulos **de treino**.

In [ ]:
arvore = DecisionTreeClassifier(max_depth=2, random_state=SEMENTE)
arvore.fit(___, ___)
print("Árvore treinada!")

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
arvore.fit(X_treino, y_treino)
```
</details>

▶️ **Rode** para desenhar a árvore.

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(arvore, feature_names=list(X.columns), class_names=["morreu", "sobreviveu"],
          filled=True, rounded=True, fontsize=11, impurity=False)
plt.show()

**Como ler o desenho** (comecem pela caixa de cima):
- A **primeira linha** de cada caixa é a **pergunta**. `sexo <= 0.5` quer dizer *"sexo vale 0?"*, ou seja, **"é homem?"** (lembrem: homem = 0, mulher = 1).
- Se a resposta é **sim (True)**, siga para a **esquerda**; se é **não (False)**, para a **direita**.
- `samples`: quantos passageiros de treino chegaram até a caixa.
- `value = [morreram, sobreviveram]`: quantos de cada tipo há na caixa.
- `class`: a resposta da árvore para quem termina ali. É a **maioria** da caixa. Laranja = morreu, azul = sobreviveu.

✍️ **Responda:**
1. Qual é a **primeira pergunta** da árvore? Faz sentido, olhando o gráfico da Etapa 2?
2. As caixas de baixo são as **respostas finais**. Escrevam as **4 regras** da árvore em português. Exemplo de formato: *"Se é homem **e** tem mais de ___ anos, então ___."*
3. A regra *"mulheres e crianças primeiro"* aparece na árvore? Onde?

**✍️ Respostas da equipe**

1.
2.
3.

## Etapa 5 · Usar a árvore para prever
Um passageiro novo segue as perguntas da árvore até uma caixa final. Para testar, usamos dois personagens do filme *Titanic* (1997): **Rose**, mulher de 17 anos na 1ª classe, e **Jack**, homem de 20 anos na 3ª classe.

▶️ **Rode.**

In [ ]:
passageiros = pd.DataFrame({
    "classe": [1, 3],
    "sexo":   [1, 0],       # 1 = mulher, 0 = homem
    "idade":  [17, 20],
    "tarifa": [100, 8],
}, index=["Rose", "Jack"])

previsto = arvore.predict(passageiros)
passageiros["previsão"] = ["sobreviveu" if p == 1 else "morreu" for p in previsto]
passageiros

Agora é a vez de vocês: inventem um passageiro (pode ser alguém da equipe embarcando em 1912!).

🧭 **Complete** com os dados do passageiro: classe (1, 2 ou 3), sexo (1 = mulher, 0 = homem), idade e tarifa (uma passagem de 3ª classe custava uns 8; de 1ª, de 30 a 500).

In [ ]:
meu_passageiro = pd.DataFrame({"classe": [___], "sexo": [___], "idade": [___], "tarifa": [___]})

resposta = arvore.predict(meu_passageiro)[0]
print("Previsão da árvore:", "sobreviveu" if resposta == 1 else "morreu")

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
meu_passageiro = pd.DataFrame({"classe": [2], "sexo": [0], "idade": [8], "tarifa": [20]})
# um menino de 8 anos na 2ª classe
```
</details>

✍️ **Responda:** sigam o caminho de Rose, de Jack e do passageiro de vocês no desenho da Etapa 4. Por quais caixas cada um passa? As previsões fazem sentido?

**✍️ Resposta da equipe:**


## Etapa 6 · Avaliar no teste
Hora da prova: a árvore prevê os passageiros de **teste**, que ela nunca viu, e comparamos com o que aconteceu de verdade.

🧭 **Complete** com o método que **faz previsões** (o par do `fit`).

In [ ]:
previsoes = arvore.___(X_teste)

acuracia = accuracy_score(y_teste, previsoes)
linha_de_base = (y_teste == 0).mean()           # o modelo preguiçoso que sempre diz "morreu"
print(f"Acurácia da árvore:            {acuracia:.1%}")
print(f"Linha de base (sempre morreu): {linha_de_base:.1%}")

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
previsoes = arvore.predict(X_teste)
```
</details>

A **matriz de confusão** mostra **que tipo** de erro o modelo comete: as **linhas** são o que aconteceu de verdade e as **colunas** são o que a árvore previu. A diagonal são os acertos.

▶️ **Rode.**

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_teste, previsoes, display_labels=["morreu", "sobreviveu"], cmap="Blues")
plt.xlabel("Previsto pela árvore")
plt.ylabel("O que aconteceu")
plt.show()

✍️ **Responda:**
1. A árvore é melhor que a linha de base? Por quantos pontos percentuais?
2. Qual erro a árvore comete **mais**: dizer que morreu quem sobreviveu, ou dizer que sobreviveu quem morreu? Olhando as 4 regras, quem são esses passageiros?

**✍️ Respostas da equipe**

1.
2.

## Etapa 7 · Qual a profundidade certa?
Uma árvore com **poucas perguntas** pode ser **simples demais** (*subajuste*). Uma árvore com **perguntas demais** pode **decorar** os passageiros de treino, um por um, e errar os novos (*sobreajuste*, Aula 07).

▶️ **Rode** para treinar árvores de várias profundidades e medir a acurácia **no treino e no teste**. `None` quer dizer *sem limite*.

In [ ]:
tabela = []
for p in [1, 2, 3, 4, 5, 8, None]:
    modelo = DecisionTreeClassifier(max_depth=p, random_state=SEMENTE).fit(X_treino, y_treino)
    tabela.append({"profundidade": str(p),
                   "acurácia no treino": accuracy_score(y_treino, modelo.predict(X_treino)),
                   "acurácia no teste": accuracy_score(y_teste, modelo.predict(X_teste))})
tabela = pd.DataFrame(tabela).round(3)

ax = tabela.plot(x="profundidade", y=["acurácia no treino", "acurácia no teste"], marker="o", figsize=(8, 4))
ax.set_ylabel("acurácia")
ax.set_title("Treino × teste conforme a árvore cresce")
plt.show()
tabela

✍️ **Responda:**
1. O que acontece com a acurácia **no treino** quando a árvore cresce sem limite? E **no teste**?
2. Qual profundidade vocês escolheriam? Lembrem: a escolha é pelo **teste**, nunca pelo treino.

**✍️ Respostas da equipe**

1.
2.

### A árvore escolhida
🧭 **Complete:** coloquem em `PROFUNDIDADE` o valor que a equipe escolheu e rodem. O gráfico mostra a **importância** de cada coluna: o quanto cada uma ajudou a árvore a separar sobreviventes de mortos.

In [ ]:
PROFUNDIDADE = ___

arvore_final = DecisionTreeClassifier(max_depth=PROFUNDIDADE, random_state=SEMENTE).fit(X_treino, y_treino)
print(f"Acurácia no teste: {accuracy_score(y_teste, arvore_final.predict(X_teste)):.1%}")

importancia = pd.Series(arvore_final.feature_importances_, index=X.columns).sort_values()
importancia.plot(kind="barh", color="tab:blue", figsize=(6, 3), title="Importância de cada coluna para a árvore")
plt.show()

<details>
<summary>💡 <b>Ver resposta</b> (tente antes!)</summary>

```python
PROFUNDIDADE = 3   # 4 também é uma boa escolha
```
</details>

✍️ **Responda:** qual coluna foi a mais importante? E a segunda? Isso combina com o que vocês viram no gráfico da Etapa 2?

**✍️ Resposta da equipe:**


> **Por que usar uma árvore de decisão?**
> - É **transparente**: dá para desenhar e ler as regras, como fizemos. Um gerente, um médico ou um juiz consegue conferir **por que** o modelo decidiu algo.
> - Funciona bem com **dados de tabela** (colunas como idade, classe, preço), que são a maioria dos dados das empresas.
> - Tem um controle fácil de entender, a **profundidade**, que deixa ver o subajuste e o sobreajuste.
>
> O cuidado: uma árvore **funda demais decora** o treino. E ela pergunta **uma coisa por vez**, o que a deixa fraca em dados como **imagens**, em que cada pixel sozinho diz pouco. É o assunto do próximo notebook.

## 🏁 Desafio extra (para quem terminar antes)
E se a árvore **não pudesse** olhar o sexo? Numa célula nova, refaçam o treino da árvore de profundidade 3 usando só as colunas `["classe", "idade", "tarifa"]` (nos dados de treino **e** de teste) e meçam a acurácia no teste. Quanto ela cai?

**✍️ Resposta da equipe:**


## Encerramento
**O que fizemos:**
1. Exploramos os dados e achamos a **linha de base** (59%).
2. Transformamos texto em número (`sexo` → 0/1) e separamos **treino e teste**.
3. Treinamos uma **árvore de decisão** e **lemos as regras** dela: a árvore redescobriu o *"mulheres e crianças primeiro"*.
4. Usamos a árvore para **prever** passageiros novos.
5. Avaliamos no teste (≈ 78% a 82% contra 59% da linha de base) e vimos o **sobreajuste** quando a árvore cresce sem limite.

➡️ **Próximo:** abram o notebook **3 · Redes neurais**.